# 0.6 이상 후보 3개: 합의와 결합

문자 TF-IDF, 단어+문자 TF-IDF, TF-IDF+E5의 동일한 924건 OOF 예측을 비교합니다. 평가 문서로 모델을 고르지 않고, 사전에 고정한 동일 가중 평균과 검토 우선 규칙만 확인합니다.

재현 명령: `python -m scripts.evaluation.candidate_ensemble`

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from matplotlib import font_manager

ROOT = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / 'scripts').is_dir()), Path.cwd().resolve())
installed = {font.name for font in font_manager.fontManager.ttflist}
plt.rcParams['font.family'] = next((f for f in ['Malgun Gothic', 'NanumGothic', 'Noto Sans CJK KR', 'DejaVu Sans'] if f in installed), 'DejaVu Sans')
plt.rcParams['axes.unicode_minus'] = False
registry = json.loads((ROOT / 'reports/model_candidates.json').read_text(encoding='utf-8'))
oof = pd.read_csv(ROOT / 'reports/model_candidate_oof.csv')
runs = {**registry['candidates'], **registry['ensembles']}
metrics = pd.DataFrame({item['name']: {key: value['fold_mean'] for key, value in item['summary'].items()} for item in runs.values()}).T
display(metrics[['macro_f1', 'accuracy', 'review_precision', 'review_recall', 'review_f1']].style.format('{:.3f}').highlight_max(axis=0, color='#b7e4c7'))

In [ ]:
agreement = pd.Series(registry['agreement'])
rates = agreement[['all_agree_rate', 'all_wrong_rate', 'at_least_one_correct_rate']].rename({'all_agree_rate': '세 모델 합의', 'all_wrong_rate': '세 모델 모두 오답', 'at_least_one_correct_rate': '하나 이상 정답'})
folds = pd.DataFrame({name: [fold['macro_f1'] for fold in item['folds']] for name, item in runs.items()})
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
rates.plot.bar(ax=axes[0], color=['#457b9d', '#e76f51', '#2a9d8f']); axes[0].set_ylim(0, 1); axes[0].set_title('후보 예측 겹침 (924건)')
(folds[['soft_vote', 'review_union']].sub(folds['tfidf_e5_hybrid'], axis=0)).plot.bar(ax=axes[1]); axes[1].axhline(0, color='black', linewidth=1); axes[1].set_title('최고 단일 후보 대비 fold별 macro F1 차이'); axes[1].set_xlabel('fold')
plt.tight_layout(); plt.show()
print(f"세 모델 합의 {agreement['all_agree_count']:.0f}/924, 모두 오답 {agreement['all_wrong_count']:.0f}/924")
print('결론: soft voting은 최고 단일 후보를 넘지 못했다. 검토 우선 규칙은 precision을 일부 내주고 recall을 올리는 운영 선택지다.')